Query
↓
Dense Retrieval (FAISS Top 5)

+

Sparse Retrieval (BM25 Top 5)

+

Web Search (DuckDuckGo Top 5)

↓

Fusion

↓

Context Creation

↓

Mistral

↓

Generated Answer

↓

Store Metrics

In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import time
import pickle
import psutil

import pandas as pd

from dotenv import load_dotenv

from duckduckgo_search import DDGS

from rank_bm25 import BM25Okapi

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_36219/3235164776.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [4]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [5]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [6]:
# ============================================================
# LOAD BM25 INDEX
# ============================================================

with open(
    "bm25_20k.pkl",
    "rb"
) as f:

    bm25 = pickle.load(f)

print(
    "BM25 Loaded Successfully"
)

BM25 Loaded Successfully


In [7]:
# ============================================================
# LOAD ALL CHUNKS
# ============================================================

with open(
    "all_chunks.pkl",
    "rb"
) as f:

    all_chunks = pickle.load(f)

print(
    "Chunks Loaded:",
    len(all_chunks)
)

Chunks Loaded: 160571


In [8]:
# ============================================================
# LOAD MISTRAL MODEL
# ============================================================

llm = ChatOllama(

    model="mistral",

    temperature=0

)

print(
    "Mistral Loaded Successfully"
)

Mistral Loaded Successfully


In [9]:
# ============================================================
# TEST MISTRAL CONNECTION
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

 Hello there! How can I assist you today?


In [10]:
queries = [

    "Harry Potter actor",

    "World Cup winner",

    "US election",

    "climate change",

    "Olympics",

    "stock market",

    "global warming",

    "terrorism",

    "education reforms",

    "covid vaccine",

    "space exploration",

    "economic crisis",

    "oil prices",

    "artificial intelligence",

    "healthcare policy",

    "renewable energy",

    "sports championship",

    "government budget",

    "scientific discovery",

    "international relations"
]

print(
    "Queries Loaded:",
    len(queries)
)

Queries Loaded: 20


In [11]:
# ============================================================
# RESULTS CONTAINER
# ============================================================

results = []

In [12]:
# ============================================================
# WEB SEARCH FUNCTION
# ============================================================

def web_search(

    query,

    max_results=5

):

    results = []

    with DDGS() as ddgs:

        search_results = list(

            ddgs.text(

                query,

                max_results=max_results

            )

        )

    for item in search_results:

        if "body" in item:

            results.append(
                item["body"]
            )

    return results

In [13]:
# ============================================================
# FUSION RAG PIPELINE
# ============================================================

for query in queries:

    print(
        f"\nProcessing Query: {query}"
    )

    start_time = time.time()

    memory_before = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    # ========================================================
    # DENSE RETRIEVAL
    # ========================================================

    dense_docs = vector_store.similarity_search(

        query,

        k=5

    )

    dense_texts = [

        doc.page_content

        for doc in dense_docs
    ]

    print(
        "Dense Results:",
        len(dense_texts)
    )

    # ========================================================
    # SPARSE RETRIEVAL
    # ========================================================

    tokenized_query = query.lower().split()

    sparse_indices = bm25.get_top_n(

        tokenized_query,

        list(
            range(
                len(all_chunks)
            )
        ),

        n=5

    )

    sparse_docs = [

        all_chunks[idx]

        for idx in sparse_indices
    ]

    print(
        "Sparse Results:",
        len(sparse_docs)
    )

    # ========================================================
    # WEB SEARCH
    # ========================================================

    web_docs = web_search(
        query,
        max_results=5
    )

    print(
        "Web Results Retrieved:",
        len(web_docs)
    )

    # ========================================================
    # FUSION
    # ========================================================

    fusion_docs = (

        dense_texts

        +

        sparse_docs

        +

        web_docs

    )

    print(
        "Fusion Documents:",
        len(fusion_docs)
    )

    # ========================================================
    # CONTEXT
    # ========================================================

    context = "\n\n".join(
        fusion_docs
    )

    print(
        "Context Length:",
        len(context)
    )

    # ========================================================
    # PROMPT
    # ========================================================

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # ========================================================
    # GENERATE RESPONSE
    # ========================================================

    response = llm.invoke(
        prompt
    )

    answer = response.content

    latency = (

        time.time()

        -

        start_time

    )

    memory_after = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    memory_used = (

        memory_after

        -

        memory_before

    )

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({

        "pipeline":
        "Fusion Mistral",

        "retrieval_method":
        "Fusion",

        "model":
        "Mistral",

        "query":
        query,

        "retrieved_count":
        len(fusion_docs),

        "retrieved_docs":
        str(fusion_docs),

        "context":
        context,

        "context_length":
        len(context),

        "generated_answer":
        answer,

        "response_length":
        len(answer),

        "latency_seconds":
        latency,

        "memory_mb":
        memory_used

    })

print(
    "\nFusion Mistral Pipeline Completed"
)


Processing Query: Harry Potter actor
Dense Results: 5
Sparse Results: 5


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_36219/2722938973.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Web Results Retrieved: 0
Fusion Documents: 10
Context Length: 4653

Processing Query: World Cup winner
Dense Results: 5
Sparse Results: 5


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_36219/2722938973.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Web Results Retrieved: 0
Fusion Documents: 10
Context Length: 4227

Processing Query: US election
Dense Results: 5
Sparse Results: 5


...

Web Results Retrieved: 0
Fusion Documents: 10
Context Length: 4527

Processing Query: scientific discovery
Dense Results: 5
Sparse Results: 5


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_36219/2722938973.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Web Results Retrieved: 0
Fusion Documents: 10
Context Length: 4146

Processing Query: international relations
Dense Results: 5
Sparse Results: 5


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_36219/2722938973.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Web Results Retrieved: 0
Fusion Documents: 10
Context Length: 3883

Fusion Mistral Pipeline Completed


In [14]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 20


,pipeline,retrieval_method,model,query,retrieved_count,retrieved_docs,context,context_length,generated_answer,response_length,latency_seconds,memory_mb
0,Fusion Mistral,Fusion,Mistral,Harry Potter actor,10,['malfoy rupert grint ron weasley and director...,malfoy rupert grint ron weasley and director d...,4653,"Daniel Radcliffe, who plays Harry Potter in t...",74,34.762885,1844.718750
1,Fusion Mistral,Fusion,Mistral,World Cup winner,10,['world cups including this summers tournament...,world cups including this summers tournament i...,4227,"The answer is not a specific person, but rath...",438,43.528732,-754.203125
2,Fusion Mistral,Fusion,Mistral,US election,15,['president since 1964 going into the election...,president since 1964 going into the election n...,5902,The provided context discusses the 2008 Unite...,1066,68.356154,584.703125
3,Fusion Mistral,Fusion,Mistral,climate change,10,['climate change let us know in the sound off ...,climate change let us know in the sound off bo...,4529,Climate change is a global issue that is happ...,821,46.329582,-676.000000
4,Fusion Mistral,Fusion,Mistral,Olympics,15,['one another thats what the olympics are abou...,one another thats what the olympics are about ...,5490,The Olympics are a global sports competition ...,426,45.980062,-926.296875


In [15]:
# ============================================================
# VALIDATE DATAFRAME
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pipeline          20 non-null     str    
 1   retrieval_method  20 non-null     str    
 2   model             20 non-null     str    
 3   query             20 non-null     str    
 4   retrieved_count   20 non-null     int64  
 5   retrieved_docs    20 non-null     str    
 6   context           20 non-null     str    
 7   context_length    20 non-null     int64  
 8   generated_answer  20 non-null     str    
 9   response_length   20 non-null     int64  
 10  latency_seconds   20 non-null     float64
 11  memory_mb         20 non-null     float64
dtypes: float64(2), int64(3), str(7)
memory usage: 203.1 KB


In [16]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "fusion_mistral_results.csv",

    index=False

)

print(
    "Fusion Mistral Results Saved Successfully"
)

Fusion Mistral Results Saved Successfully
